## Setup

In [ ]:
import pymupdf
import re

from pathlib import Path

import polars as pl
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# Create project root
def find_project_root(marker: str = '.git') -> Path:
    '''Climbs up folders starting from CWD until it finds the root.'''
    current = Path.cwd().resolve()

    # Loop upward until hitting the project root (where current == its own parent)
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent

PROJECT_ROOT = find_project_root(marker='.git')
PDF_DIR = PROJECT_ROOT / 'data' / 'raw' / 'records'

## Analyze Project Descriptions

Standard Permit documents differs from PSD/GHGPSD documents. Furthermore the document structures differ among the same permit types depending on the time period so the entire first page is passed to the agent. 

The main model used is `claude-sonnet-5` as `claude-haiku-4-5` demonstrates many mistakes while `claude-opus-4-8` demonstrates strong performance but consumes considerably more tokens.

In [ ]:
# LLM project description analysis

async def analyze_proj_desc(cover_text: str) -> dict:
    schema = {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "site_name": {"type": "string"},
                "county": {"type": "string"},
                "is_data_center": {"type": "boolean"},
                "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                "capacity_MW": {"type": "number"}
            },
            "required": ["site_name", "county", "is_data_center", "confidence", "capacity_MW"]
        }
    }

    stderr_lines = []

    options = ClaudeAgentOptions(
        system_prompt = '''
            You are a document classifier for TCEQ air permit filings. Given a project description, determine whether it describes a gas-fired power generation facility built to power an onsite or immediately-adjacent data center / compute facility behind the meter (i.e., generation dedicated to the data center's load, not routed through the grid).

            Examples NOT qualifying:
            - Crypto-mining facilities (regardless of power source)
            - Grid-interconnected generation, even if colocated with other industrial loads
            - Generation whose stated purpose is oilfield/production support (compressors, artificial lift, etc.) rather than compute load
            - Generation serving mixed/unspecified loads with no data center mentioned

            For ambigious descriptions which could possibly suggest one, perform a web search. Examples include "onsite generation", "behind-the-meter", etc. without naming what load it serves. Do three rounds of independent searches for such

            Confidence rubric:
            - "high": the document explicitly states the generation powers a data center/compute facility
            - "medium": the document implies it (e.g., site name, operator, or context strongly suggests it) but doesn't state it outright
            - "low": classification relies on external web search or is a close judgment call

            If the facility qualifies, extract operating capacity in MW from the document; use -1 if not stated.
        ''',
        max_turns=25,
        model="claude-sonnet-5",
        output_format=schema,
        stderr=stderr_lines.append,
    )

    prompt = f"Project Description: {cover_text}"

    messages = []
    async for message in query(prompt=prompt, options=options):
        messages.append(message)
        if isinstance(message, ResultMessage):
            if message.is_error:
                raise RuntimeError(f"Result error ({message.subtype}): {message.errors}")
            if message.structured_output is not None:
                return message.structured_output
            raise RuntimeError(f"ResultMessage had no structured_output (subtype={message.subtype})")

    print("No ResultMessage received. Full message sequence:")
    for m in messages:
        print(" -", m)
    if stderr_lines:
        print("CLI stderr:")
        for line in stderr_lines:
            print(" ", line)
    raise RuntimeError("No result message received")

In [19]:
pdf_paths = list(PDF_DIR.glob("*.pdf"))

def extract_section(prefix: str, suffix: str, text):
    pattern = rf"{prefix}(.*?){suffix}"

    return re.search(pattern, text, re.DOTALL).group(1)

# Consumption: 10% of Pro Plan for 30 documents.
proj_infos = []
for path in pdf_paths:
    cover_text = pymupdf.open(path)[0].get_text()

    proj_info = await analyze_proj_desc(cover_text)
    proj_info["path"] = f"{path}"
    proj_infos.append(proj_info)

site_records = pl.DataFrame(
    {
        key: [
            data[key]
            for data in proj_infos
        ]
        for key in proj_infos[0].keys()
    },
    strict=False
)

data_centers = site_records.filter(
    (pl.col("is_data_center") == True)
    | (
        (pl.col("is_data_center") == False)
        & (pl.col("confidence") == "low")
    )
)

data_centers

site_name,county,is_data_center,confidence,capacity_MW,path
str,str,bool,str,f64,object
"""Circe Energy Data Centers S. M…","""Ward""",true,"""high""",400.0,/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_183234-409633_Permits_Public_20260605_Technical Review_8486732_.pdf
"""Coyote Spring Agr1""","""Reeves""",false,"""low""",-1.0,/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_176054-410845_Permits_Public_20260623_Technical Review_8516829_.pdf
"""GW Ranch Energy Center""","""Pecos""",true,"""high""",5000.0,/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_AirPermits-396366_Permits_Public_20260121_Technical Review_8201547_.pdf
"""Kilby Power Plant""","""Reeves""",false,"""low""",-1.0,/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_AirPermits-399548_Permits_Public_20260415_Technical Review_8350931_.pdf
"""Rock House Draw Generating Sta…","""Pecos""",true,"""high""",576.0,/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_181521-398180_Permits_Public_20251023_Technical Review_7973167_.pdf
"""Pecos Power Generation""","""Reeves""",true,"""high""",478.0,/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_179308-389727_Permits_Public_20250305_Technical Review_7630740_.pdf
"""Comanche Creek Generating Stat…","""Pecos""",false,"""low""",-1.0,/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_176519-374640_Permits_Public_20240606_Technical Review_7107107_.pdf
"""Xenergy Monahans 1""","""Ward""",true,"""high""",118.98,/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_174494-410474_Permits_Public_20260604_Technical Review_8477114_.pdf
"""Featherwood Energies Pecos Pla…","""Reeves""",true,"""high""",931.0,/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_182541-402447_Permits_Public_20260108_Technical Review_8180573_.pdf


## Checkpoints

In the instance where there are insufficient tokens but you want to save and load progress.

In [ ]:
# Save data
def save_data(df: pl.DataFrame, path: str):
    df.write_csv(path)

CHECKPOINT_NAME = "classified-list.csv"
SAVE_PATH = PROJECT_ROOT / "data" / "processed" / CHECKPOINT_NAME 

save_data(site_records, SAVE_PATH)

In [ ]:
# Load data
def load_data(path: str) -> pl.DataFrame:
    return pl.read_csv(path)

site_records = load_data(SAVE_PATH)

## Determine Best Sections

Some pdfs can be dozens of pages long. In that case, it is inefficient to pass the entirety of the document. Since the document structures are somewhat inconsistent, sentence transformers are used to retrieve sections with higher relevance.

In [51]:
def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 200) -> list[dict[int, str]]:
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start: end]
        chunks.append(chunk)
        start += (chunk_size - overlap)

    return chunks


def retrieve_best_pages(path: str, model: SentenceTransformer, query: str, top_k=3):
    doc_text = "\n".join([page.get_text() for page in pymupdf.open(path)])
    chunks_text = chunk_text(doc_text)

    query_vector = model.encode([query])
    chunk_vectors = model.encode([chunk for chunk in chunks_text])

    similarities = cosine_similarity(query_vector, chunk_vectors)[0]

    scored_chunks = []
    for i, score in enumerate(similarities):
        scored_chunks.append({
            "text": chunks_text[i],
            "score": score
        })

    scored_chunks.sort(key=lambda x: x["score"], reverse=True)
    return scored_chunks[:top_k]


# % ---- Quick reset ----
data_centers = site_records.filter(
    (pl.col("is_data_center") == True)
    | (
        (pl.col("is_data_center") == False)
        & (pl.col("confidence") == "low")
    )
)
# % ---- Quick reset ----

model = SentenceTransformer('all-MiniLM-L6-v2')
query = "Engine or turbine specifications including originel equipment manufacturer (OEM) and model number."

best_pages = []
for path in data_centers.get_column("path"):
    best_pages = retrieve_best_pages(path, model, query)
    print(path, best_pages)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4995.41it/s]


/Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/records/AIR NSR_183234-409633_Permits_Public_20260605_Technical Review_8486732_.pdf [{'text': 'cumventing \nmajor NSR permitting.  (30 TAC §116.110)\n* No\nStandard Permit Rules Check\nComments\nType of unit being authorized\n*Engine(s)\nFees\n*   ≥ 1 MW     $900\nFee Reference / Receipt Number\n832203 / 582EA000735837\nCombined Heat and Power (if taking credit)\nN/A\nFuel\n*  Natural gas (<10 grains total sulfur per 100 dscf)\nMW rating:\nPer unit: 2 MW\nSitewide:400 MW\n1\n\nElectric Generating Unit Standard Permit \nTechnical Review\nRegistration No. 183234\nRegulated Entity No. RN112406442\nPage 2\nNOx emission limitation (in lb/MWh)\n     * West Texas\n           *>300 hrs/yr – 3.11 lb/MWhr\nFacility Nox Emissions\nEngine Model – Cummins HSK78G\nAllowable Emission Rate (lbs/hr): 6.22 lb/hr\nActual (lbs/hr): 0.20\nMaintenance, Startup, and Shutdown\nMaintenance activities will be authorized either by PBR or De \nMinimis. 